# Advanced Mushroom Classification Training
**Danish Fungi-style training with:**
- FocalLoss for class imbalance
- Mixup / CutMix augmentation
- EMA (Exponential Moving Average)
- Gradient clipping
- Strong augmentations

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip install timm tqdm scikit-learn pillow pandas --quiet

In [ ]:
import os
import sys
import math
from copy import deepcopy
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm
import timm
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration

In [ ]:
# ============ CONFIGURE PATHS HERE ============
CONFIG = {
    # Data paths (adjust to your Google Drive structure)
    'csv_path': '/content/drive/MyDrive/DF20M-metadata/DF20M-train_metadata_PROD.csv',
    'data_dir': '/content/drive/MyDrive/DF20M',
    'checkpoint_dir': '/content/drive/MyDrive/checkpoints',
    
    # Model
    'model_name': 'vit_small_patch16_224',  # or 'vit_base_patch16_224'
    'pretrained': True,
    'img_size': 224,
    
    # Training
    'epochs': 30,
    'batch_size': 32,
    'lr': 1e-4,
    'weight_decay': 0.05,
    'num_workers': 2,
    
    # Loss: 'ce', 'focal', 'label_smoothing'
    'loss_type': 'focal',
    'focal_gamma': 2.0,
    'label_smoothing': 0.1,
    
    # Augmentation
    'mixup_alpha': 0.8,
    'cutmix_alpha': 1.0,
    'mixup_prob': 0.5,
    
    # Regularization
    'use_ema': True,
    'ema_decay': 0.9999,
    'clip_grad': 1.0,
    
    # AMP
    'use_amp': True,
}

os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)
print("Configuration loaded!")

## Loss Functions

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance."""
    def __init__(self, alpha=1.0, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


class LabelSmoothingCrossEntropy(nn.Module):
    """Cross entropy with label smoothing."""
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        self.confidence = 1.0 - smoothing

    def forward(self, inputs, targets):
        logprobs = F.log_softmax(inputs, dim=-1)
        nll_loss = -logprobs.gather(dim=-1, index=targets.unsqueeze(1)).squeeze(1)
        smooth_loss = -logprobs.mean(dim=-1)
        loss = self.confidence * nll_loss + self.smoothing * smooth_loss
        return loss.mean()

## EMA (Exponential Moving Average)

In [ ]:
class ModelEMA:
    """Exponential Moving Average of model weights."""
    def __init__(self, model, decay=0.9999, device=None):
        self.ema = deepcopy(model)
        self.ema.eval()
        self.decay = decay
        self.device = device
        if self.device is not None:
            self.ema.to(device=device)
        for p in self.ema.parameters():
            p.requires_grad_(False)

    def update(self, model):
        with torch.no_grad():
            for ema_p, model_p in zip(self.ema.parameters(), model.parameters()):
                ema_p.data.mul_(self.decay).add_(model_p.data, alpha=1 - self.decay)

    def state_dict(self):
        return self.ema.state_dict()

    def load_state_dict(self, state_dict):
        self.ema.load_state_dict(state_dict)

## Mixup / CutMix

In [ ]:
def mixup_data(x, y, alpha=0.8):
    """Mixup: Creates mixed inputs and targets."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def cutmix_data(x, y, alpha=1.0):
    """CutMix: Cuts and pastes patches."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size, _, H, W = x.size()
    index = torch.randperm(batch_size, device=x.device)
    
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)
    
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    
    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)
    
    x[:, :, bby1:bby2, bbx1:bbx2] = x[index, :, bby1:bby2, bbx1:bbx2]
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (W * H))
    
    y_a, y_b = y, y[index]
    return x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Compute loss for mixup/cutmix."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

## Dataset & Transforms

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def get_transforms(mode='train', img_size=224):
    if mode == 'train':
        return transforms.Compose([
            transforms.Resize((img_size + 32, img_size + 32)),
            transforms.RandomCrop((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.2),
            transforms.RandomRotation(30),
            transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
            transforms.RandomGrayscale(p=0.1),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            transforms.RandomErasing(p=0.25, scale=(0.02, 0.33))
        ])
    else:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        ])


class MushroomDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, species_to_id=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        
        if species_to_id is None:
            unique_species = sorted(self.df['species'].unique())
            self.species_to_id = {sp: idx for idx, sp in enumerate(unique_species)}
        else:
            self.species_to_id = species_to_id
        
        self.num_classes = len(self.species_to_id)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, row['image_path'])
        
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        
        label = self.species_to_id[row['species']]
        return {'image': image, 'label': label}

## Create Dataloaders

In [ ]:
# Load and split data
df = pd.read_csv(CONFIG['csv_path'])
df = df.dropna(subset=['species']).reset_index(drop=True)

# Create label mapping
unique_species = sorted(df['species'].unique())
species_to_id = {sp: idx for idx, sp in enumerate(unique_species)}
num_classes = len(species_to_id)

print(f"Total samples: {len(df)}")
print(f"Number of classes: {num_classes}")

# Split by observation ID
unique_ids = df['gbifID'].unique()
train_ids, val_ids = train_test_split(unique_ids, test_size=0.2, random_state=42)

train_df = df[df['gbifID'].isin(train_ids)].reset_index(drop=True)
val_df = df[df['gbifID'].isin(val_ids)].reset_index(drop=True)

print(f"Train samples: {len(train_df)}")
print(f"Val samples: {len(val_df)}")

# Create datasets
train_ds = MushroomDataset(
    train_df, 
    CONFIG['data_dir'], 
    transform=get_transforms('train', CONFIG['img_size']),
    species_to_id=species_to_id
)

val_ds = MushroomDataset(
    val_df, 
    CONFIG['data_dir'], 
    transform=get_transforms('val', CONFIG['img_size']),
    species_to_id=species_to_id
)

# Create dataloaders
train_loader = DataLoader(
    train_ds, batch_size=CONFIG['batch_size'], shuffle=True,
    num_workers=CONFIG['num_workers'], pin_memory=True
)

val_loader = DataLoader(
    val_ds, batch_size=CONFIG['batch_size'], shuffle=False,
    num_workers=CONFIG['num_workers'], pin_memory=True
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## Create Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create model
model = timm.create_model(
    CONFIG['model_name'],
    pretrained=CONFIG['pretrained'],
    num_classes=num_classes
)
model = model.to(device)

# EMA
ema = None
if CONFIG['use_ema']:
    ema = ModelEMA(model, decay=CONFIG['ema_decay'], device=device)
    print(f"EMA enabled with decay={CONFIG['ema_decay']}")

# Loss function
if CONFIG['loss_type'] == 'focal':
    criterion = FocalLoss(gamma=CONFIG['focal_gamma'])
    print(f"Using FocalLoss (gamma={CONFIG['focal_gamma']})")
elif CONFIG['loss_type'] == 'label_smoothing':
    criterion = LabelSmoothingCrossEntropy(smoothing=CONFIG['label_smoothing'])
    print(f"Using LabelSmoothingCrossEntropy (smoothing={CONFIG['label_smoothing']})")
else:
    criterion = nn.CrossEntropyLoss()
    print("Using CrossEntropyLoss")

# Optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'], eta_min=CONFIG['lr'] * 0.01)

# AMP scaler
scaler = GradScaler() if CONFIG['use_amp'] else None
if CONFIG['use_amp']:
    print("Mixed precision (AMP) enabled")

print(f"\nModel: {CONFIG['model_name']}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training Loop

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device, scaler, ema, config):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    use_mixup = config['mixup_alpha'] > 0 or config['cutmix_alpha'] > 0
    
    pbar = tqdm(loader, desc="Training")
    for batch in pbar:
        images = batch['image'].to(device)
        labels = batch['label'].to(device)
        
        # Apply Mixup or CutMix
        if use_mixup and np.random.random() < config['mixup_prob']:
            if config['cutmix_alpha'] > 0 and np.random.random() > 0.5:
                images, labels_a, labels_b, lam = cutmix_data(images, labels, config['cutmix_alpha'])
            else:
                images, labels_a, labels_b, lam = mixup_data(images, labels, config['mixup_alpha'])
            mixed = True
        else:
            mixed = False
        
        optimizer.zero_grad()
        
        if scaler is not None:
            with autocast():
                outputs = model(images)
                if mixed:
                    loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
                else:
                    loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            
            if config['clip_grad'] is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config['clip_grad'])
            
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            if mixed:
                loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
            else:
                loss = criterion(outputs, labels)
            loss.backward()
            
            if config['clip_grad'] is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), config['clip_grad'])
            
            optimizer.step()
        
        if ema is not None:
            ema.update(model)
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        if not mixed:
            correct += predicted.eq(labels).sum().item()
        else:
            correct += predicted.eq(labels_a).sum().item()
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.2f}%'})
    
    return running_loss / len(loader), 100. * correct / total


@torch.no_grad()
def validate(model, loader, criterion, device, use_amp=False):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch in tqdm(loader, desc="Validating"):
        images = batch['image'].to(device)
        labels = batch['label'].to(device)
        
        if use_amp:
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return running_loss / len(loader), 100. * correct / total

In [ ]:
# Training
best_val_acc = 0.0

print("="*60)
print("Starting Advanced Training")
print("="*60)

for epoch in range(CONFIG['epochs']):
    print(f"\nEpoch {epoch+1}/{CONFIG['epochs']}")
    print("-" * 40)
    
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device, scaler, ema, CONFIG
    )
    
    # Validate with EMA model if available
    if ema is not None:
        val_loss, val_acc = validate(ema.ema, val_loader, criterion, device, CONFIG['use_amp'])
        print(f"(EMA) Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    else:
        val_loss, val_acc = validate(model, val_loader, criterion, device, CONFIG['use_amp'])
    
    scheduler.step()
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    print(f"LR: {optimizer.param_groups[0]['lr']:.2e}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_dict = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'num_classes': num_classes,
            'species_to_id': species_to_id,
            'config': CONFIG,
        }
        if ema:
            save_dict['ema_state_dict'] = ema.state_dict()
        
        torch.save(save_dict, os.path.join(CONFIG['checkpoint_dir'], 'best_model.pth'))
        print(f"Saved best model with val acc: {val_acc:.2f}%")

print(f"\n" + "="*60)
print(f"Training complete! Best validation accuracy: {best_val_acc:.2f}%")
print("="*60)

## Download Best Model

In [ ]:
# Download model from Colab
from google.colab import files
files.download(os.path.join(CONFIG['checkpoint_dir'], 'best_model.pth'))